# L03 · The Smallest RL Problem: Bandits

## Goal

- distinguish exploration from exploitation
- compute regret
- update action estimates from sampled rewards

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L03:toy:42").hexdigest()
print(f"lesson=L03 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L03 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:590ac6402f99d22939b7ae82eae358e15f3fe446668384655cf87d7b911bfaea data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: probability/gradients → **bandits** → MDPs

$$Q_{n+1}(a)=Q_n(a)+\frac{1}{N_n(a)}\left(R_n-Q_n(a)\right)$$

A bandit has no state transitions; each step chooses one arm. Exploitation follows the current best estimate, while exploration tests uncertain arms. Regret accumulates expected-reward gaps, not realized sample losses.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** With all estimates initialized to zero, will pure greedy always discover the best arm? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>No. Early randomness and tie-breaking can trap it. A small epsilon breaks that failure path.</details>

In [2]:
from rl_study.envs import BernoulliBandit
def run_bandit(epsilon):
    env = BernoulliBandit(horizon=120)
    env.reset(seed=42)
    counts, estimates, regret = [0] * 5, [0.0] * 5, 0.0
    rng = random.Random(42)
    for step in range(120):
        explore = rng.random() < epsilon
        action = rng.randrange(5) if explore else max(range(5), key=estimates.__getitem__)
        result = env.step(action)
        counts[action] += 1
        estimates[action] += (result.reward - estimates[action]) / counts[action]
        regret += float(result.info["expected_regret"])
    return regret, counts, estimates
greedy_run = run_bandit(0.0)
exploring_run = run_bandit(0.1)
print({"greedy_regret": round(greedy_run[0], 2),
       "epsilon_regret": round(exploring_run[0], 2),
       "epsilon_counts": exploring_run[1]})

{'greedy_regret': 84.0, 'epsilon_regret': 28.25, 'epsilon_counts': [4, 3, 59, 1, 53]}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** The incremental mean reproduces the sample mean without storing rewards. UCB and Thompson sampling are alternatives, but epsilon-greedy makes the role of exploration most transparent.

**Common trap:** Using sampled rewards for regret can make a lucky bad arm appear to have negative regret. Accumulate the environment's expected regret separately. Regression tests: `test_bandit_seed_is_deterministic`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert sum(exploring_run[1]) == 120 and exploring_run[0] >= 0.0
print("checks=passed")

checks=passed


**Recall:** How can exploration reduce long-run regret while lowering immediate reward? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** For the fixed seed, epsilon=0.1 had lower cumulative regret than pure greedy. This is one environment's failure case, not a universal superiority claim.
- Executable checks: `test_bandit_seed_is_deterministic`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L04 adds states and transitions, moving to MDPs where actions change future rewards.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`